# Libraries 

In [ ]:
# ==============================
# Core libraries
# ==============================
import numpy as np
import pandas as pd
import time
import warnings
import multiprocessing
from datetime import date, datetime, timedelta

# Use all but one CPU core for parallel processing
num_cores = max(multiprocessing.cpu_count() - 1, 1)
print("Using", num_cores, "cores for parallel processing.")

warnings.filterwarnings("ignore")


# ==============================
# Visualisation
# ==============================
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8")  # optional, just to make plots look nicer


# ==============================
# Statistical utilities (EDA, tests)
# ==============================
import scipy.stats as stats
from scipy.stats import chi2, chi2_contingency, f_oneway

# Statsmodels (OLS, ANOVA, etc.)
import statsmodels.api as sm
from statsmodels.formula.api import ols


# ==============================
# Preprocessing & feature engineering
# ==============================
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    RobustScaler,
    OneHotEncoder
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Optional: dimensionality reduction
from sklearn.decomposition import PCA


# ==============================
# Modelling algorithms (base models)
# ==============================

# Linear models / GLM-style
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

# Tree-based and ensemble models
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    BaggingRegressor,
    StackingRegressor,
    VotingRegressor
)

from sklearn.tree import DecisionTreeRegressor

# Distance-based & kernel-based models
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

# Neural network regressor
from sklearn.neural_network import MLPRegressor


# ==============================
# Gradient boosting libraries (external)
# ==============================
# XGBoost
try:
    import xgboost as xgb
    xgb_available = True
    print("XGBoost available.")
except ImportError:
    xgb_available = False
    print("XGBoost NOT available (install xgboost if you want to use it).")

# LightGBM
try:
    import lightgbm as lgb
    lgb_available = True
    print("LightGBM available.")
except ImportError:
    lgb_available = False
    print("LightGBM NOT available (install lightgbm if you want to use it).")

# CatBoost (optional; often slower but nice to try)
try:
    import catboost as cb
    cb_available = True
    print("CatBoost available.")
except ImportError:
    cb_available = False
    print("CatBoost NOT available (install catboost if you want to use it).")


# ==============================
# Model evaluation & selection
# ==============================
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ==============================
# Model interpretation tools
# ==============================
from sklearn.inspection import (
    permutation_importance,
    PartialDependenceDisplay
)

# If you later want SHAP, you can add:
# import shap


# Loading the Data & Basic Inspection

## Loading the Data

In [ ]:
# Importing the training dataset
train = pd.read_csv("ML_WP_data/train.csv")
# Checking the table
train.head()
# Calculate total number of NaN values in the DataFrame
total_train_nans = train.isna().sum().sum()

# Display the total count of NaN values
print("Total NaN values in the DataFrame:", total_train_nans)
print(f"Train shape: {train.shape}")
display(train.head())

# ------------------------------
# Helper: missingness summary (train only)
# ------------------------------

def missing_summary(df, sort_by="Percent_Missing", ascending=False):
    """
    Build a table with:
    - Data type
    - Number of missing values
    - Percentage of missing values
    - Basic descriptive stats (mean, std, min, 25%, 50%, 75%, max) for numeric cols
    """

    n_rows = len(df)

    # Core missing-value info
    miss = df.isna().sum()
    miss = miss[miss > 0]  # keep only variables with at least one missing

    summary = pd.DataFrame({
        "Data_Type": df[miss.index].dtypes.astype(str),
        "Missing_Values": miss,
        "Percent_Missing": (miss / n_rows * 100).round(2)
    })

    # Descriptive stats for numeric columns
    numeric_cols = df[miss.index].select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        desc = df[numeric_cols].describe().T[
            ["mean", "std", "min", "25%", "50%", "75%", "max"]
        ]
        summary = summary.join(desc, how="left")

    # Sort and print total
    summary = summary.sort_values(sort_by, ascending=ascending)
    print(f"Total variables with missing values: {summary.shape[0]}")

    return summary

# Compute missingness summary for train
missing_train = missing_summary(train)
display(missing_train.head(70))  # show first 70 rows; adjust as needed